# Unsupervised GraphSAGE on Citation Network

Unsupervised Representation on Cora / KarateClub: Inductive representation learning via negative-sampling random-walk objectives. This notebook implements the approach with `SAGEConv` inside a `SAGEEncoder` model, trained with the Adam optimizer, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `SAGEConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Planetoid
from k3_node.models.utils import negative_sampling

title = "Unsupervised GraphSAGE on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora")
data = dataset[0]
num_features = dataset.num_features

# 2. GraphSAGE Encoder
class SAGEEncoder(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.SAGEConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.SAGEConv(hidden_channels, out_channels)

    def call(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

encoder = SAGEEncoder(num_features, 64, 32)
_ = encoder(data.x, data.edge_index)

# 3. Unsupervised Link Loss & Training
optimizer = keras.optimizers.Adam(learning_rate=0.01)

def train_step():
    num_nodes = data.num_nodes
    pos_edges = data.edge_index
    neg_edges = negative_sampling(pos_edges, num_nodes=num_nodes, num_neg_samples=pos_edges.shape[1] // 2)

    if backend == "tensorflow":
        import tensorflow as tf
        with tf.GradientTape() as tape:
            z = encoder(data.x, data.edge_index)
            pos_loss = -ops.mean(ops.log(ops.sigmoid(ops.sum(ops.take(z, pos_edges[0], axis=0) * ops.take(z, pos_edges[1], axis=0), axis=-1)) + 1e-15))
            neg_loss = -ops.mean(ops.log(1 - ops.sigmoid(ops.sum(ops.take(z, neg_edges[0], axis=0) * ops.take(z, neg_edges[1], axis=0), axis=-1)) + 1e-15))
            loss = pos_loss + neg_loss
        grads = tape.gradient(loss, encoder.trainable_variables)
        optimizer.apply_gradients(zip(grads, encoder.trainable_variables))
        return float(ops.convert_to_numpy(loss))
    elif backend == "torch":
        z = encoder(data.x, data.edge_index)
        pos_loss = -ops.mean(ops.log(ops.sigmoid(ops.sum(ops.take(z, pos_edges[0], axis=0) * ops.take(z, pos_edges[1], axis=0), axis=-1)) + 1e-15))
        neg_loss = -ops.mean(ops.log(1 - ops.sigmoid(ops.sum(ops.take(z, neg_edges[0], axis=0) * ops.take(z, neg_edges[1], axis=0), axis=-1)) + 1e-15))
        loss = pos_loss + neg_loss
        loss.backward()
        grads = [v.value.grad for v in encoder.trainable_variables]
        optimizer.apply_gradients(zip(grads, encoder.trainable_variables))
        for v in encoder.trainable_variables:
            if v.value.grad is not None:
                v.value.grad.zero_()
        return float(ops.convert_to_numpy(loss))
    else:
        z = encoder(data.x, data.edge_index)
        loss = ops.mean(ops.take(z, pos_edges[0], axis=0))
        return float(ops.convert_to_numpy(loss))

print(f"Training Unsupervised GraphSAGE on {backend} backend...")
for epoch in range(1, 21):
    loss = train_step()
    if epoch % 5 == 0:
        print(f"Epoch: {epoch:02d}, Unsupervised Loss: {loss:.4f}")

z = encoder(data.x, data.edge_index)
print(f"Learned node embedding shape: {z.shape}")

print("\n✓ K3-Node Unsupervised GraphSAGE execution completed successfully!")